In [1]:
from typing import Iterable
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle
import ipywidgets as widgets
from IPython.display import display, clear_output
from utils import get_itr_index, index_to_type, load_data, align_track_hpc

In [2]:
_CPOS_COLOR = {
    "CAB": "#1f77b4",
    "CBA": "#1f77b4",

    "ACB": "#ff7f0e",
    "BCA": "#ff7f0e",

    "ABC": "#d62728",
    "BAC": "#d62728",
}


def _infer_n_neurons(
    data: dict,
    ana_tt: Iterable[str],
    ana_bt: Iterable[str],
) -> int:
    for tt_idx, bt_idx in get_itr_index(ana_tt, ana_bt):
        raw = data["aligned_firing"][tt_idx, bt_idx]
        if raw is not None:
            return raw.shape[0]
    raise ValueError("No valid condition found.")


def plot_hpc_neuron_ab_ba(
    data: dict,
    neuron_id: int,
    ana_bt: Iterable[str] = ("Correct",),
    *,
    show_sem: bool = True,
    show_zones: bool = True,
    space_unit: float = 2.0,
    alpha: float = 0.9,
    lw: float = 1.8,
    figsize=(10, 3.2),
):
    tracks_ab = ["CAB", "ACB", "ABC"]
    tracks_ba = ["CBA", "BCA", "BAC"]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    def plot_one_panel(ax, track_names, panel_title):
        found_any = False

        for tt_name in track_names:
            for tt_idx, bt_idx in get_itr_index([tt_name], ana_bt):
                fr_mean = data["aligned_firing"][tt_idx, bt_idx]
                fr_std = data["firing_std"][tt_idx, bt_idx]

                if fr_mean is None:
                    continue

                if not (0 <= neuron_id < fr_mean.shape[0]):
                    raise ValueError(
                        f"neuron_id {neuron_id} out of range, valid range: [0, {fr_mean.shape[0]-1}]"
                    )

                y = fr_mean[neuron_id]
                yerr = fr_std[neuron_id] if fr_std is not None else np.zeros_like(y)
                x = np.arange(y.shape[0])

                color = _CPOS_COLOR[tt_name]

                ax.plot(
                    x,
                    y,
                    color=color,
                    linewidth=lw,
                    alpha=alpha,
                )

                if show_sem:
                    ax.fill_between(
                        x,
                        y - yerr,
                        y + yerr,
                        color=color,
                        alpha=0.18,
                    )

                found_any = True

        if show_zones:

            zones = data["zones"]
            if zones is None:
                return

            selected = {
                0: "#1f77b4",  # zone 1
                2: "#ff7f0e",  # zone 3
                4: "#d62728",  # zone 5
            }

            h_frac = 0.05

            for zi, color in selected.items():
                if zi >= len(zones):
                    continue

                row = zones[zi]
                if row is None or len(row) < 2:
                    continue

                start_x, end_x = float(row[0]), float(row[1])

                ax.axvline(start_x, linestyle="--", color="k", linewidth=0.8)
                ax.axvline(end_x, linestyle="--", color="k", linewidth=0.8)

                rect = Rectangle(
                    (start_x, 1.0 - h_frac),
                    end_x - start_x,
                    h_frac,
                    transform=ax.get_xaxis_transform(),
                    facecolor=color,
                    edgecolor="none",
                    clip_on=False,
                    zorder=6,
                )
                ax.add_patch(rect)

        ax.set_title(panel_title, fontsize=13)
        ax.set_xlabel("Position (cm)", fontsize=12)
        ax.tick_params(axis="both", labelsize=11)
        ax.xaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, pos: f"{x * space_unit:g}")
        )
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.xaxis.set_ticks_position("bottom")
        ax.yaxis.set_ticks_position("left")

        return found_any

    ok_ab = plot_one_panel(axes[0], tracks_ab, "AB")
    ok_ba = plot_one_panel(axes[1], tracks_ba, "BA")

    if not (ok_ab or ok_ba):
        raise ValueError("No valid firing array found for this neuron.")

    axes[0].set_ylabel("Deconvolved activity", fontsize=12)

    if "cell_ids" in data and data["cell_ids"] is not None and neuron_id < len(data["cell_ids"]):
        fig.suptitle(f"Neuron #{neuron_id} (cell id: {data['cell_ids'][neuron_id]})", fontsize=14, y=1.03)
    else:
        fig.suptitle(f"Neuron #{neuron_id}", fontsize=14, y=1.03)

    plt.tight_layout()
    plt.show()

    return fig, axes


def browse_hpc_neurons_ab_ba(
    data: dict,
    ana_bt: Iterable[str] = ("Correct",),
    *,
    gaussian_sigma: float = 2,
    show_sem: bool = True,
    show_zones: bool = True,
    space_unit: float = 2.0,
    alpha: float = 0.9,
    lw: float = 1.8,
    figsize=(10, 3.2),
    id_min: int = 0,
    id_max: int | None = None,
):
    data = align_track_hpc(
        data,
        gaussian_sigma=gaussian_sigma,
    )
    
    n_neurons = _infer_n_neurons(
        data,
        ana_tt=["*"],
        ana_bt=ana_bt
    )

    if id_max is None:
        id_max = n_neurons - 1

    if id_min < 0 or id_max >= n_neurons or id_min > id_max:
        raise ValueError(f"Invalid id range: [{id_min}, {id_max}], valid range is [0, {n_neurons - 1}]")

    slider = widgets.IntSlider(
        value=id_min,
        min=id_min,
        max=id_max,
        step=1,
        description="Neuron",
        continuous_update=False,
    )
    btn_prev = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="80px"))
    btn_next = widgets.Button(description="Next ▶", layout=widgets.Layout(width="80px"))
    out = widgets.Output()

    def render(_=None):
        with out:
            clear_output(wait=True)
            plot_hpc_neuron_ab_ba(
                data,
                neuron_id=slider.value,
                ana_bt=ana_bt,
                show_sem=show_sem,
                show_zones=show_zones,
                space_unit=space_unit,
                alpha=alpha,
                lw=lw,
                figsize=figsize,
            )

    def on_prev(_):
        slider.value = max(slider.min, slider.value - 1)

    def on_next(_):
        slider.value = min(slider.max, slider.value + 1)

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    slider.observe(render, names="value")

    display(widgets.HBox([btn_prev, slider, btn_next]), out)
    render()

In [3]:
data = load_data("../../data/HPC_2p/HP07/neuro_type_saveHP07_17_2025-03-11.mat")

In [ ]:
browse_hpc_neurons_ab_ba(
    data,
    ana_bt=["Correct"],
)

Output()